# PDF Parsing Agent: Multimodal Extraction & Evaluation

This project implements a sophisticated, multi-agent RAG (Retrieval-Augmented Generation) pipeline designed specifically for complex financial documents. Using **LangGraph** and **Gemini 2.5 Flash**, the system coordinates between "specialist" agents to classify pages, crop charts, transcribe tables, and self-correct via automated QA loops.

---

## 🤖 Component Functions (Nodes)

The system is built as a state-managed graph where each node performs a specific task within the `AgentState`.

### 1. Core Utilities
* **`AgentState`**: A `TypedDict` that maintains the "memory" of the agent, including the current page index, accumulated chunks, classification results, and a spillover buffer for multi-page tables.
* **`get_page_base64`**: Converts a specific PDF page into a high-resolution JPEG (300 DPI) and encodes it to base64 for the Gemini vision model.

### 2. Classification & Routing
* **`page_classifier_node`**: Analyzes the page image to determine its type: `table_of_contents`, `standard_text`, `contains_chart`, or `junk`. It also identifies when the document enters a new section or subsection to maintain hierarchical metadata.

### 3. Visual Extraction Agents
* **`chart_counter_node`**: Acts as a supervisor that scans the page to count the exact number of distinct charts, graphs, or data tables.
* **`visual_graph_extractor_node`**: Uses the count from the supervisor to locate and return bounding boxes `[ymin, xmin, ymax, xmax]` for every chart on the page. It then uses `PIL` to crop these regions into standalone images.
* **`draft_chart_node`**: Transcribes the cropped chart images into highly descriptive Markdown, ensuring every row, column, and footnote is captured without summarization.

### 4. Text & Metadata Extraction
* **`text_drafter_node`**: Extracts all text from the page and formats it into clean Markdown.
* **`date_extractor_node`**: Scans the drafted text to extract all significant dates as a comma-separated list.
* **`people_extractor_node`**: Identifies and extracts names of key individuals mentioned in the text.

### 5. Quality Assurance (QA) Loops
* **`text_qa_node` / `visual_qa_node`**: Auditors that compare the drafted Markdown against the original page image to ensure no data was missed or misread.
* **`date_qa_node` / `people_qa_node`**: Verify that extracted entities are actually present in the text and haven't been hallucinated.

### 6. Finalization
* **`chunk_and_advance_node`**: Splits the verified text into smaller chunks using a `RecursiveCharacterTextSplitter`. It attaches hierarchical metadata (Section/Subsection) and the extracted entities (Dates/People) before advancing to the next page.

---

## 🔄 The Workflow Flow

The `StateGraph` logic orchestrates the agents in a recursive loop until the entire document is processed:

1.  **Entry Point**: `classifier`.
2.  **Logic Branching**:
    * **Junk Pages**: Routed to `skip_page` and then back to the classifier for the next page.
    * **Charts/Tables**: Routed through `chart_counter` $\rightarrow$ `visual_graph_extractor` $\rightarrow$ `draft_chart` $\rightarrow$ `visual_qa`.
    * **Standard Text**: Routed directly to `text_drafter` $\rightarrow$ `text_qa`.
3.  **Refinement**: If a QA node detects an error, it sends the state back to the respective drafter with feedback for a retry (limited by `attempts`).
4.  **Enrichment**: Once the text is verified, it passes through `date_extractor` and `people_extractor`.
5.  **Completion**: The `chunk_and_advance` node saves the results and checks if more pages remain.

---

## 📥 Input & 📤 Output

### Input
* **PDF File**: A path to a document (e.g., `Earnings Release (FY26 Q4).pdf`).
* **Initial State**: A dictionary containing configuration like `total_pages`, `PROJECT_ID`, and `LOCATION`.

### Output: `extracted_chunks.json`
The system produces a JSON array of multimodal chunks. Each chunk contains:
* `content`: The enriched Markdown text or table.
* `type`: Either `stitched_chart` or `contextual_text`.
* `metadata`:
    * `source_page`: The original page number.
    * `section` / `subsection`: Hierarchical position in the document.
    * `extracted_dates` / `extracted_people`: Lists of entities found on that page.
    * `visual_graph_base64`: (Optional) The cropped image of a chart relevant to that chunk.

---
## Example Data comes from Walmart's FY26 Q4 Earnings Report
[See Link here.](https://stock.walmart.com/_assets/_461d6b46a29d437b51015f942ff9bb4e/walmart/db/938/9972/earnings_release/Earnings+Release+(FY26+Q4).pdf)

## 🧪 Example Test Suite

The system includes a verification suite of 20 questions used to evaluate the accuracy of the extracted data.

| Question | Ground Truth Answer (Example) |
| :--- | :--- |
| What was Walmart's total revenue for Q4 fiscal 2026? | $190.7 billion, an increase of 5.6%. |
| How much did global eCommerce sales grow? | Grew by 24%. |
| What was the size of the new share repurchase authorization? | $30 billion. |
| What is the guidance for capital expenditures in FY 2027? | Approximately 3.5% of net sales. |

### Questions must be in JSON Format
```
test_suite = [
    {
      "question": "What was Walmart's total revenue for the fourth quarter of fiscal 2026?",
      "answer": "Walmart's total revenue for the fourth quarter was $190.7 billion, representing an increase of 5.6%[cite: 979]."
    },
    {
      "question": "How much did Walmart's global eCommerce sales grow in the fourth quarter?",
      "answer": "Global eCommerce sales grew by 24%, led by store-fulfilled pickup & delivery and marketplace[cite: 980]."
    }]
```
---



## 🚀 Production Roadmap

To move this system into a production environment, consider the following steps:

1.  **Asynchronous Processing**: Implement `async` nodes in LangGraph to process multiple pages in parallel, significantly reducing processing time for long documents.
2.  **Vector Database Integration**: Replace the JSON export with a direct stream into a production vector store (e.g., Vertex AI Search or Pinecone) to enable immediate querying.
3.  **Blob Storage for Images**: Instead of storing large base64 strings in metadata, upload cropped charts to Google Cloud Storage (GCS) and store only the authenticated URL.
4.  **Observability**: Integrate **LangSmith** to monitor the graph's execution, track token costs, and identify which pages trigger the most "QA Fail" loops.
5.  **Human-in-the-loop**: Add a manual review step for pages where the agent reaches its maximum retry attempts without passing QA.



In [1]:
!pip install -qU google-genai faiss-cpu langchain-community poppler-utils langgraph langchain-google-genai pdf2image pydantic langchain-core Pillow PyMuPDF
!apt-get update
!apt-get install -y poppler-utils

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.6/760.6 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.7/240.7 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.

In [10]:
import fitz # PyMuPDF is great for quickly getting page counts
import sys
import base64
import json
import io
import PIL.Image
from io import BytesIO
from typing import TypedDict, List, Dict, Any, Optional
from pydantic import BaseModel, Field
from pdf2image import convert_from_path
from langgraph.graph import StateGraph, END
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from google import genai
from google.genai import types
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_text_splitters import RecursiveCharacterTextSplitter

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()

# --- Initialization ---
PROJECT_ID = ""  #@param {type:"string"}
LOCATION = "us-central1" #@param {type:"string"}

# Use the updated LangChain Google integration
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    project=PROJECT_ID,
    location=LOCATION
)

In [11]:
# Extractor Schemas
class ChartCount(BaseModel):
    count: int = Field(description="The exact number of distinct charts, graphs, or data tables on the page.")

class ChartDetection(BaseModel):
    has_graphs: bool
    boxes_2d: List[List[int]] = Field(
        description="List of bounding boxes [ymin, xmin, ymax, xmax] for EVERY chart or table on the page.",
        default_factory=list
    )

# Generalized Agent State
class AgentState(TypedDict):
    pdf_path: str
    total_pages: int
    current_page_idx: int

    spillover_buffer: List[str]
    final_chunks: List[Dict]

    classification: Dict
    draft_chunk: str
    feedback: str
    attempts: int
    expected_chart_count: int
    extracted_graph_images: List[str]
    active_table_headers: str

    # Generalized Hierarchy Trackers
    current_section: str
    current_subsection: str
    document_title: str

    # --- NEW: Date & People Tracking ---
    page_dates: List[str]
    date_attempts: int
    date_feedback: str

    page_people: List[str]
    people_attempts: int
    people_feedback: str

# Helper to load single page on demand to prevent memory bloat
def get_page_base64(pdf_path: str, page_num: int, dpi: int = 300) -> str:
    # pdf2image allows extracting specific pages (1-indexed)
    pages = convert_from_path(pdf_path, dpi=dpi, first_page=page_num, last_page=page_num)
    buffer = BytesIO()
    pages[0].save(buffer, format="JPEG", quality=90)
    return base64.b64encode(buffer.getvalue()).decode('utf-8')

In [12]:
def page_classifier_node(state: AgentState):
    page_num = state["current_page_idx"] + 1
    print(f"\n--- Classifying Page {page_num} ---")

    current_page_b64 = get_page_base64(state["pdf_path"], page_num)

    prompt = """Analyze this page from a financial or corporate document. Respond in strict JSON format:
    {
        "page_type": "table_of_contents" | "standard_text" | "contains_chart" | "junk",
        "chart_spills_over": true | false,
        "is_table_continuation": true | false,
        "starts_new_section": "Name of the Section if this page explicitly introduces a major new chapter/section, else null",
        "starts_new_subsection": "Name of the Subsection if a new sub-topic is introduced, else null"
    }
    CRITICAL INSTRUCTIONS:
    1. Classify as "junk" ONLY IF the page is purely legal disclaimers, totally blank, or just a cover photo with no data.
    2. 'is_table_continuation' is true ONLY if a data table starts immediately at the very top without a new title."""

    message = HumanMessage(content=[
        {"type": "text", "text": prompt},
        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{current_page_b64}"}}
    ])

    try:
        response = llm.invoke([message])
        classification = json.loads(response.content.strip("```json\n").strip("```"))
    except Exception:
        classification = {"page_type": "standard_text", "chart_spills_over": False, "is_table_continuation": False}

    updates = {"classification": classification}

    # Update Generic Hierarchy
    if classification.get("starts_new_section"):
        updates["current_section"] = classification["starts_new_section"]
        updates["current_subsection"] = "Introduction" # Reset subsection on new section
        print(f"   📍 Entered New Section: {updates['current_section']}")

    if classification.get("starts_new_subsection"):
        updates["current_subsection"] = classification["starts_new_subsection"]
        print(f"   📄 Entered New Subsection: {updates['current_subsection']}")

    return updates

In [13]:
def visual_qa_node(state: AgentState):
    print("-> QA Reviewing Chart Draft...")
    draft = state["draft_chunk"]
    attempts = state.get("attempts", 0)
    page_num = state["current_page_idx"] + 1

    if attempts >= 2 or "===CHART_SEPARATOR===" in draft or len(draft) > 50:
        section = state.get("current_section", "General Document")
        subsection = state.get("current_subsection", "Overview")

        individual_drafts = [d.strip() for d in draft.split("===CHART_SEPARATOR===") if d.strip()]
        images = state.get("extracted_graph_images", [])

        new_chunks = []
        for i, d in enumerate(individual_drafts):
            chunk_meta = {
                "source_page": page_num,
                "section": section,
                "subsection": subsection
            }
            if i < len(images):
                chunk_meta["visual_graph_base64"] = images[i]

            chart_context = f"This financial chart is located within the '{section}' section, specifically under '{subsection}'.\n\n"
            new_chunks.append({"content": chart_context + d, "metadata": chunk_meta, "type": "stitched_chart"})

        return {
            "final_chunks": state["final_chunks"] + new_chunks,
            "spillover_buffer": [],
            "attempts": 0, "feedback": "", "draft_chunk": "",
            "extracted_graph_images": []
        }
    return {"feedback": "Draft incomplete or formatting missed.", "attempts": attempts + 1}

def text_qa_node(state: AgentState):
    print("-> QA Reviewing Text Draft...")
    page_num = state["current_page_idx"] + 1
    current_page_b64 = get_page_base64(state["pdf_path"], page_num)
    draft = state["draft_chunk"]
    attempts = state.get("attempts", 0)

    prompt = f"You are a strict QA Auditor. Compare the Draft Text to the original image. Did the drafter miss any paragraphs, footnotes, or misread numbers?\n\nDraft Text:\n{draft}\n\nIf perfect, respond ONLY with 'PASS'. If errors exist, respond with 'FAIL:' followed by specific missed details."

    response = llm.invoke([HumanMessage(content=[
        {"type": "text", "text": prompt},
        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{current_page_b64}"}}
    ])])

    if response.content.strip().startswith("PASS") or attempts >= 2:
        # Pass control to date extraction, do not chunk or advance page yet
        return {"attempts": 0, "feedback": ""}

    return {"feedback": response.content.strip(), "attempts": attempts + 1}

# --- NEW: Date Extractor & QA ---
def date_extractor_node(state: AgentState):
    print("-> Extracting Dates...")
    draft = state["draft_chunk"]
    critique = f"\nQA REJECTED LAST DRAFT. Fix these issues:\n{state.get('date_feedback')}\n" if state.get("date_feedback") else ""

    prompt = f"Extract all significant dates from the following text. Return them as a comma-separated list. If no dates are found, return 'NONE'.{critique}\n\nText:\n{draft}"
    response = llm.invoke([HumanMessage(content=[{"type": "text", "text": prompt}])])

    dates_str = response.content.strip()
    date_list = [d.strip() for d in dates_str.split(",")] if dates_str.upper() != "NONE" else []
    return {"page_dates": date_list}

def date_qa_node(state: AgentState):
    print("-> QA Reviewing Dates...")
    attempts = state.get("date_attempts", 0)
    draft = state["draft_chunk"]
    dates = state.get("page_dates", [])

    prompt = f"You are a strict QA Auditor. Verify that the following extracted dates are correct and present in the text.\n\nExtracted Dates: {dates}\n\nText:\n{draft}\n\nIf perfect or if no dates are legitimately present, respond ONLY with 'PASS'. If dates are missed or hallucinated, respond with 'FAIL:' followed by the specific error."
    response = llm.invoke([HumanMessage(content=[{"type": "text", "text": prompt}])])

    if response.content.strip().startswith("PASS") or attempts >= 2:
        return {"date_feedback": "", "date_attempts": 0}
    return {"date_feedback": response.content.strip(), "date_attempts": attempts + 1}

# --- NEW: People Extractor & QA ---
def people_extractor_node(state: AgentState):
    print("-> Extracting Key People...")
    draft = state["draft_chunk"]
    critique = f"\nQA REJECTED LAST DRAFT. Fix these issues:\n{state.get('people_feedback')}\n" if state.get("people_feedback") else ""

    prompt = f"Extract all key people (names of individuals) from the following text. Return them as a comma-separated list. If no people are found, return 'NONE'.{critique}\n\nText:\n{draft}"
    response = llm.invoke([HumanMessage(content=[{"type": "text", "text": prompt}])])

    people_str = response.content.strip()
    people_list = [p.strip() for p in people_str.split(",")] if people_str.upper() != "NONE" else []
    return {"page_people": people_list}

def people_qa_node(state: AgentState):
    print("-> QA Reviewing People...")
    attempts = state.get("people_attempts", 0)
    draft = state["draft_chunk"]
    people = state.get("page_people", [])

    prompt = f"You are a strict QA Auditor. Verify that the following extracted people are correct and present in the text.\n\nExtracted People: {people}\n\nText:\n{draft}\n\nIf perfect or if no people are legitimately present, respond ONLY with 'PASS'. If people are missed or hallucinated, respond with 'FAIL:' followed by the specific error."
    response = llm.invoke([HumanMessage(content=[{"type": "text", "text": prompt}])])

    if response.content.strip().startswith("PASS") or attempts >= 2:
        return {"people_feedback": "", "people_attempts": 0}
    return {"people_feedback": response.content.strip(), "people_attempts": attempts + 1}

# --- REFACTORED: Finalize and Chunk ---
def chunk_and_advance_node(state: AgentState):
    print("-> Finalizing chunks with new metadata and advancing page...")
    page_num = state["current_page_idx"] + 1
    draft = state["draft_chunk"]
    section = state.get("current_section", "General Document")
    subsection = state.get("current_subsection", "Overview")
    dates = state.get("page_dates", [])
    people = state.get("page_people", [])

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    paragraphs = text_splitter.split_text(draft)

    new_chunks = []
    for para in paragraphs:
        if len(para.strip()) < 50: continue

        context_prompt = f"""You are an AI preparing financial document chunks for a vector database.
        HIERARCHY: Section: {section} -> Subsection: {subsection}

        Full page context: <context>{draft}</context>
        Specific chunk: <chunk>{para}</chunk>

        Write a succinct context (1-2 sentences) situating this chunk within its specific section and subsection. Respond ONLY with the contextual prefix."""

        ctx_response = llm.invoke(context_prompt)
        enriched_content = f"{ctx_response.content.strip()}\n\n{para}"

        meta = {
            "source_page": page_num,
            "section": section,
            "subsection": subsection,
            "extracted_dates": dates,    # Injected extracted dates
            "extracted_people": people   # Injected extracted people
        }
        new_chunks.append({"content": enriched_content, "metadata": meta, "type": "contextual_text"})

    return {
        "final_chunks": state["final_chunks"] + new_chunks,
        "current_page_idx": page_num, # Advance the page index
        "draft_chunk": "",
        "page_dates": [],             # Reset accumulators for next page
        "page_people": []
    }

In [14]:
def chart_counter_node(state: AgentState):
    page_num = state["current_page_idx"] + 1
    print(f"-> Supervisor Agent: Counting charts on page {page_num}...")

    current_page_b64 = get_page_base64(state["pdf_path"], page_num)
    structured_llm = llm.with_structured_output(ChartCount)

    prompt = "Carefully scan this page. Count the exact number of distinct charts, graphs, and data tables. Ignore purely decorative images."
    message = HumanMessage(content=[{"type": "text", "text": prompt}, {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{current_page_b64}"}}])

    try:
        result = structured_llm.invoke([message])
        count = result.count
        print(f"   Supervisor counted {count} chart(s).")
    except Exception as e:
        print(f"   Counter Failed: {e}. Defaulting to 1.")
        count = 1
    return {"expected_chart_count": count}

def visual_graph_extractor_node(state: AgentState):
    expected_count = state.get("expected_chart_count", 1)
    page_num = state["current_page_idx"] + 1
    print(f"-> Extractor Agent: Hunting for exactly {expected_count} bounding box(es)...")

    current_page_b64 = get_page_base64(state["pdf_path"], page_num)
    structured_llm = llm.with_structured_output(ChartDetection)

    prompt = f"The supervisor detected EXACTLY {expected_count} chart(s)/table(s) on this page. Locate EVERY SINGLE ONE and return their bounding boxes."
    message = HumanMessage(content=[{"type": "text", "text": prompt}, {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{current_page_b64}"}}])

    images = []
    try:
        detection = structured_llm.invoke([message])
        if detection.has_graphs and detection.boxes_2d:
            img_data = base64.b64decode(current_page_b64)
            img = PIL.Image.open(io.BytesIO(img_data))
            width, height = img.size

            for box in detection.boxes_2d:
                ymin, xmin, ymax, xmax = box
                # Crop the image using the bounding box
                cropped_img = img.crop(((xmin / 1000.0) * width, (ymin / 1000.0) * height, (xmax / 1000.0) * width, (ymax / 1000.0) * height))
                buffered = io.BytesIO()
                cropped_img.save(buffered, format="JPEG", quality=90)
                images.append(base64.b64encode(buffered.getvalue()).decode('utf-8'))
            print(f"   Successfully cropped {len(images)}/{expected_count} graphs from this page!")
    except Exception as e:
        print(f"   Graph Extraction Failed: {e}")
    return {"extracted_graph_images": images}

def draft_chart_node(state: AgentState):
    attempts = state.get('attempts', 0)
    page_num = state["current_page_idx"] + 1
    print(f"-> Drafting Chart Markdown (Attempt {attempts + 1})...")

    current_page_b64 = get_page_base64(state["pdf_path"], page_num)
    buffer = state.get("spillover_buffer", []) + [current_page_b64]

    critique = f"\nQA REJECTED YOUR LAST DRAFT. Fix these issues:\n{state['feedback']}\n" if state.get("feedback") else ""
    is_cont = state.get("classification", {}).get("is_table_continuation", False)
    saved_headers = state.get("active_table_headers", "")
    header_instruction = f"\nCRITICAL: This is a continuation. Use these headers:\n{saved_headers}\n" if (is_cont and saved_headers) else ""

    prompt = f"""Extract EVERY chart/table on this page into a highly descriptive Markdown format.
    CRITICAL INSTRUCTIONS:
    1. DO NOT SUMMARIZE. You must transcribe every single row, column, number, label, and footnote exactly as it appears.
    2. If there are multiple charts, separate each chart's markdown using exactly this string on its own line: ===CHART_SEPARATOR===
    {critique}{header_instruction}"""

    content = [{"type": "text", "text": prompt}]
    for img in buffer:
        content.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img}"}})

    response = llm.invoke([HumanMessage(content=content)])
    return {"draft_chunk": response.content}

def text_drafter_node(state: AgentState):
    attempts = state.get('attempts', 0)
    page_num = state["current_page_idx"] + 1
    print(f"-> Drafting Standard Text (Attempt {attempts + 1})...")

    current_page_b64 = get_page_base64(state["pdf_path"], page_num)

    critique = f"\nQA REJECTED LAST DRAFT. Fix these issues:\n{state['feedback']}\n" if state.get("feedback") else ""
    prompt = f"Extract all text from this page. Format it beautifully in Markdown. Do not summarize.{critique}"

    response = llm.invoke([HumanMessage(content=[
        {"type": "text", "text": prompt},
        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{current_page_b64}"}}
    ])])

    return {"draft_chunk": response.content}

In [17]:

def continue_or_end(state: AgentState) -> str:
    # Check against total_pages instead of the old pdf_pages array
    if state["current_page_idx"] >= state["total_pages"]:
        print("\n--- All pages processed! Ending workflow. ---")
        return END
    return "classifier"

def route_page(state: AgentState) -> str:
    if state["current_page_idx"] >= state["total_pages"]:
        return END

    page_type = state.get("classification", {}).get("page_type")

    if page_type == "junk":
        return "skip_page"
    elif page_type == "table_of_contents":
        return "text_drafter"
    elif page_type == "contains_chart":
        return "buffer_page" if state.get("classification", {}).get("chart_spills_over") else "chart_counter"

    return "text_drafter"

# Keep your existing continue_or_end and route_page functions...

def route_text_qa(state: AgentState) -> str:
    if state.get("feedback") != "":
        return "text_drafter"
    return "date_extractor" # Changed to go to dates instead of ending

def route_date_qa(state: AgentState) -> str:
    if state.get("date_feedback") != "":
        return "date_extractor"
    return "people_extractor"

def route_people_qa(state: AgentState) -> str:
    if state.get("people_feedback") != "":
        return "people_extractor"
    return "chunk_and_advance"

# --- Build and Compile the Graph ---
workflow = StateGraph(AgentState)

# Add all standard nodes
workflow.add_node("classifier", page_classifier_node)
workflow.add_node("skip_page", lambda state: {"current_page_idx": state["current_page_idx"] + 1})
workflow.add_node("buffer_page", lambda state: {"spillover_buffer": state["spillover_buffer"] + [get_page_base64(state["pdf_path"], state["current_page_idx"] + 1)], "current_page_idx": state["current_page_idx"] + 1})
workflow.add_node("chart_counter", chart_counter_node)
workflow.add_node("visual_graph_extractor", visual_graph_extractor_node)
workflow.add_node("draft_chart", draft_chart_node)
workflow.add_node("visual_qa", visual_qa_node)
workflow.add_node("text_drafter", text_drafter_node)
workflow.add_node("text_qa", text_qa_node)

# Add new nodes
workflow.add_node("date_extractor", date_extractor_node)
workflow.add_node("date_qa", date_qa_node)
workflow.add_node("people_extractor", people_extractor_node)
workflow.add_node("people_qa", people_qa_node)
workflow.add_node("chunk_and_advance", chunk_and_advance_node)

# Set Entry Point
workflow.set_entry_point("classifier")

# Chart edges
workflow.add_conditional_edges("classifier", route_page, {
    "skip_page": "skip_page",
    "chart_counter": "chart_counter",
    "buffer_page": "buffer_page",
    "text_drafter": "text_drafter",
    END: END
})
workflow.add_edge("chart_counter", "visual_graph_extractor")
workflow.add_edge("visual_graph_extractor", "draft_chart")
workflow.add_edge("draft_chart", "visual_qa")
workflow.add_edge("visual_qa", "text_drafter")
workflow.add_edge("text_drafter", "text_qa")

# New Text/Data Routing Edges
workflow.add_conditional_edges("text_qa", route_text_qa, {
    "text_drafter": "text_drafter",
    "date_extractor": "date_extractor"
})

workflow.add_edge("date_extractor", "date_qa")
workflow.add_conditional_edges("date_qa", route_date_qa, {
    "date_extractor": "date_extractor",
    "people_extractor": "people_extractor"
})

workflow.add_edge("people_extractor", "people_qa")
workflow.add_conditional_edges("people_qa", route_people_qa, {
    "people_extractor": "people_extractor",
    "chunk_and_advance": "chunk_and_advance"
})

workflow.add_conditional_edges("chunk_and_advance", continue_or_end, {
    "classifier": "classifier",
    END: END
})

workflow.add_conditional_edges("buffer_page", continue_or_end, {"classifier": "classifier", END: END})
workflow.add_conditional_edges("skip_page", continue_or_end, {"classifier": "classifier", END: END})

chunking_app = workflow.compile()
print("Graph compiled successfully! 'chunking_app' is now ready to invoke.")

Graph compiled successfully! 'chunking_app' is now ready to invoke.


In [18]:
PDF_PATH = "/content/Earnings Release (FY26 Q4).pdf"
doc = fitz.open(PDF_PATH)
total_pages = 19

initial_state = {
    "pdf_path": PDF_PATH,
    "total_pages": total_pages,
    "current_page_idx": 0,
    "spillover_buffer": [],
    "final_chunks": [],
    "classification": {},
    "draft_chunk": "",
    "feedback": "",
    "attempts": 0,
    "expected_chart_count": 0,
    "extracted_graph_images": [],
    "active_table_headers": "",
    "current_section": "Executive Summary",
    "current_subsection": "Overview",
    "document_title": "Financial Report",

    # Initialize New Extraction State
    "page_dates": [],
    "date_attempts": 0,
    "date_feedback": "",
    "page_people": [],
    "people_attempts": 0,
    "people_feedback": ""
}

# Increased the recursion limit multiplier from 5 to 15 to account for the new agent loops
final_state = chunking_app.invoke(initial_state, {"recursion_limit": total_pages * 15})


--- Classifying Page 1 ---
   📍 Entered New Section: Walmart reports Q4 results
-> Drafting Standard Text (Attempt 1)...
-> QA Reviewing Text Draft...
-> Extracting Dates...
-> QA Reviewing Dates...
-> Extracting Key People...
-> QA Reviewing People...
-> Finalizing chunks with new metadata and advancing page...

--- Classifying Page 2 ---
   📍 Entered New Section: Key Financial Metrics
   📄 Entered New Subsection: Balance Sheet and Liquidity
-> Supervisor Agent: Counting charts on page 2...
   Supervisor counted 14 chart(s).
-> Extractor Agent: Hunting for exactly 14 bounding box(es)...
   Successfully cropped 14/14 graphs from this page!
-> Drafting Chart Markdown (Attempt 1)...
-> QA Reviewing Chart Draft...
-> Drafting Standard Text (Attempt 1)...
-> QA Reviewing Text Draft...
-> Extracting Dates...
-> QA Reviewing Dates...
-> Extracting Key People...
-> QA Reviewing People...
-> Finalizing chunks with new metadata and advancing page...

--- Classifying Page 3 ---
   📍 Entered New

In [19]:

# Define the file path
CHUNKS_PATH = "extracted_chunks.json"

# Save the chunks from the final state of your graph
print(f"Saving {len(final_state['final_chunks'])} chunks to {CHUNKS_PATH}...")
with open(CHUNKS_PATH, "w") as f:
    json.dump(final_state["final_chunks"], f, indent=4)

print("Saved successfully! You are now ready to run the evaluation script.")

Saving 114 chunks to extracted_chunks.json...
Saved successfully! You are now ready to run the evaluation script.


In [21]:
# Initialize the new standard client
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

EMBED_MODEL = "gemini-embedding-2-preview"
JUDGE_MODEL = "gemini-2.5-flash-lite"

# --- 2. Define Your Generic Test Suite ---
# Populate this with ground-truth questions and answers based on the PDF you ingested.
test_suite = [
    {
      "question": "What was Walmart's total revenue for the fourth quarter of fiscal 2026?",
      "answer": "Walmart's total revenue for the fourth quarter was $190.7 billion, representing an increase of 5.6%[cite: 979]."
    },
    {
      "question": "How much did Walmart's global eCommerce sales grow in the fourth quarter?",
      "answer": "Global eCommerce sales grew by 24%, led by store-fulfilled pickup & delivery and marketplace[cite: 980]."
    },
    {
      "question": "What was the size of the new share repurchase authorization announced by the company?",
      "answer": "The company announced a new $30 billion share repurchase authorization[cite: 972]."
    },
    {
      "question": "What was Walmart's total revenue for the full fiscal year 2026?",
      "answer": "Walmart's total revenue for the full year was $713.2 billion, an increase of 4.7%[cite: 986]."
    },
    {
      "question": "According to the balance Sheet and liquidity highlights, what was Walmart's operating cash flow?",
      "answer": "Operating cash flow was $41.6 billion, which was an increase of $5.1 billion[cite: 1078]."
    },
    {
      "question": "What is Walmart's guidance for net sales growth in constant currency for fiscal year 2027?",
      "answer": "Walmart expects net sales to grow between 3.5% and 4.5% in constant currency for fiscal year 2027[cite: 976, 1169]."
    },
    {
      "question": "What were the net sales for the Walmart U.S. segment in the fourth quarter of fiscal 2026?",
      "answer": "The net sales for the Walmart U.S. segment were $123.5 billion in the fourth quarter[cite: 1090]."
    },
    {
      "question": "How much did Sam's Club U.S. eCommerce sales grow in the fourth quarter?",
      "answer": "Sam's Club U.S. eCommerce sales were up 23%, with continued strong growth in club-fulfilled pickup & delivery[cite: 1153]."
    },
    {
      "question": "How much did Walmart's global advertising business grow during the full fiscal year?",
      "answer": "The global advertising business grew 46% to nearly $6.4 billion, including VIZIO[cite: 988, 989]."
    },
    {
      "question": "How many stores and associates does Walmart have globally?",
      "answer": "Each week, approximately 280 million customers and members visit more than 10,900 stores and numerous eCommerce websites, and the company employs approximately 2.1 million associates worldwide[cite: 1181, 1182]."
    },
    {
      "question": "What was Walmart's adjusted earnings per share (EPS) for the fourth quarter?",
      "answer": "Walmart's adjusted EPS for the fourth quarter was $0.74[cite: 971]."
    },
    {
      "question": "What was Walmart's free cash flow for the fiscal year ended January 31, 2026?",
      "answer": "Free cash flow was $14.9 billion, representing an increase of $2.3 billion[cite: 1078]."
    },
    {
      "question": "What were the net sales for Walmart International in the fourth quarter of fiscal 2026?",
      "answer": "Net sales for Walmart International in the fourth quarter were $35.9 billion[cite: 1100]."
    },
    {
      "question": "What were Walmart's Return on Assets (ROA) and Return on Investment (ROI) for the trailing twelve months ended January 31, 2026?",
      "answer": "Walmart's ROA was 8.2% and its ROI was 15.1% for the trailing twelve months ended January 31, 2026[cite: 1363, 1365]."
    },
    {
      "question": "How much did Walmart's global inventory increase by the end of the fourth quarter?",
      "answer": "Global inventory increased by 4.3%, reaching $58.9 billion[cite: 1080]."
    },
    {
      "question": "What is Walmart's new increased annual dividend amount per share?",
      "answer": "Walmart increased its annual dividend to $0.99 per share[cite: 992]."
    },
    {
      "question": "By what percentage did Walmart Connect sales grow in the U.S. during the fourth quarter?",
      "answer": "Walmart Connect sales in the U.S. were up 41%[cite: 981]."
    },
    {
      "question": "What is Walmart's guidance for capital expenditures in fiscal year 2027?",
      "answer": "Capital expenditures are expected to be approximately 3.5% of net sales[cite: 1169]."
    },
    {
      "question": "What were the total net sales for Sam's Club U.S. for the full fiscal year 2026?",
      "answer": "Total net sales for Sam's Club U.S. for the full fiscal year were $93.0 billion[cite: 1115]."
    },
    {
      "question": "What was Walmart's total debt as reported in the Balance Sheet and Liquidity highlights?",
      "answer": "Walmart's total debt was $51.5 billion[cite: 1077]."
    }
]
# --- 3. Load Data & Build Vector Index ---
print("Loading Contextual Chunks...")
with open("extracted_chunks.json", "r") as f:
    valid_chunks = json.load(f)

corpus_embeddings = []

print(f"Embedding {len(valid_chunks)} multimodal chunks...")
for chunk in valid_chunks:
    text_content = chunk.get("content", "")
    if not text_content.strip(): continue

    embed_inputs = [text_content]
    meta = chunk.get("metadata", {})

    # If the chunk has a cropped chart, attach it to the embedding payload
    if meta.get("visual_graph_base64"):
        image_bytes = base64.b64decode(meta["visual_graph_base64"])
        embed_inputs.append(types.Part.from_bytes(data=image_bytes, mime_type="image/jpeg"))

    res = client.models.embed_content(
        model=EMBED_MODEL,
        contents=embed_inputs,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT", output_dimensionality=768)
    )
    corpus_embeddings.append(res.embeddings[0].values)

embedding_matrix = np.array(corpus_embeddings)
print("Vector Index built successfully!\n")

# --- 4. Run LLM Judge Evaluation ---
print(f"🚀 Starting Top-3 Evaluation for {len(test_suite)} questions...\n")

passed_count = 0
failed_questions = []

for i, item in enumerate(test_suite):
    question, ground_truth = item["question"], item["answer"]

    # A: Search Vector DB
    q_res = client.models.embed_content(
        model=EMBED_MODEL,
        contents=[question],
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY", output_dimensionality=768)
    )
    q_vec = np.array(q_res.embeddings[0].values).reshape(1, -1)

    # Grab the Top 5 best matching chunks
    best_indices = np.argsort(cosine_similarity(q_vec, embedding_matrix)[0])[::-1][:5]

    combined_context = ""
    judge_contents = []

    # B: Build a combined context block
    for rank, idx in enumerate(best_indices):
        chunk = valid_chunks[idx]
        meta = chunk.get("metadata", {})

        # Generalized metadata fields
        page = meta.get('source_page', 'Unknown')
        section = meta.get('section', 'Unknown')
        subsection = meta.get('subsection', 'Unknown')

        combined_context += f"\n--- RETRIEVED CHUNK {rank+1} (Page {page} | {section} -> {subsection}) ---\n{chunk['content']}\n"

        # If any of these top chunks are charts, add their images to the Judge's vision payload
        if meta.get("visual_graph_base64"):
            img_bytes = base64.b64decode(meta["visual_graph_base64"])
            judge_contents.append(types.Part.from_bytes(data=img_bytes, mime_type="image/jpeg"))

    # C: Prompt the Judge
    judge_prompt_text = f"""You are an expert grading assistant evaluating a Retrieval-Augmented Generation (RAG) system for complex financial documents.

    QUESTION: {question}
    GROUND TRUTH ANSWER: {ground_truth}

    RETRIEVED CONTEXT (Top 3 Matches):
    {combined_context}

    TASK: Does the RETRIEVED CONTEXT contain sufficient and accurate information to answer the QUESTION in agreement with the GROUND TRUTH ANSWER?
    *Note: Be lenient if a page number is slightly off due to title pages.*

    Respond strictly in this format:
    RESULT: [PASS or FAIL]
    EXPLANATION: [One sentence explaining why it passed or failed]
    """

    # Insert the text prompt at the beginning of the list, followed by any images we found
    judge_contents.insert(0, judge_prompt_text)

    eval_response = client.models.generate_content(model=JUDGE_MODEL, contents=judge_contents)
    result_text = eval_response.text.strip()

    if "RESULT: PASS" in result_text.upper():
        passed_count += 1
        print(f"✅ Q{i+1}: PASS | {question}")
    else:
        print(f"❌ Q{i+1}: FAIL | {question}")
        print(f"   {result_text}\n")
        failed_questions.append(question)

# --- 5. Final Results ---
accuracy = (passed_count / len(test_suite)) * 100 if len(test_suite) > 0 else 0
print("\n" + "="*60)
print(f"🏆 FINAL EVALUATION SCORE: {passed_count}/{len(test_suite)} ({accuracy:.1f}%)")
print("="*60)

Loading Contextual Chunks...
Embedding 114 multimodal chunks...
Vector Index built successfully!

🚀 Starting Top-3 Evaluation for 20 questions...

✅ Q1: PASS | What was Walmart's total revenue for the fourth quarter of fiscal 2026?
✅ Q2: PASS | How much did Walmart's global eCommerce sales grow in the fourth quarter?
✅ Q3: PASS | What was the size of the new share repurchase authorization announced by the company?
✅ Q4: PASS | What was Walmart's total revenue for the full fiscal year 2026?
❌ Q5: FAIL | According to the balance Sheet and liquidity highlights, what was Walmart's operating cash flow?
   RESULT: FAIL
EXPLANATION: The retrieved context discusses revenue, operating income, and various other financial metrics but does not contain any information about Walmart's operating cash flow.

✅ Q6: PASS | What is Walmart's guidance for net sales growth in constant currency for fiscal year 2027?
✅ Q7: PASS | What were the net sales for the Walmart U.S. segment in the fourth quarter of f